In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv()

True

In [4]:
from langchain_openai import ChatOpenAI
chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

In [5]:
from langchain_core.prompts import ChatPromptTemplate
promt_txt = """{query}"""
prompt = ChatPromptTemplate.from_template(promt_txt)


In [13]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda

In [10]:
user_query ='Explain gen ai in 10 words'
lcChain = (RunnablePassthrough() | prompt | chatgpt)
response = lcChain.invoke({"query":user_query})

In [11]:
response.content

'Generative AI creates content using algorithms and machine learning techniques.'

In [17]:
# Lets say we have process the input query and only need to pass the first token in the input to LLM, so we have written python function for that,
# now we can not call this function directly inside LangChain chain for the we need to use RunnableLambda
def clean_query(data: dict):
    text = data["input"]  
    return text.split()[2]

In [18]:
llmchain = (RunnablePassthrough.assign(query=RunnableLambda(clean_query)) | prompt | chatgpt) 
user_query1 = "GenAI and ML"
response = llmchain.invoke({'input':user_query1})

In [19]:
response.content

"It looks like you're interested in machine learning (ML). How can I assist you with that? Are you looking for information on specific algorithms, applications, tools, or something else?"

In [20]:
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [22]:
docs = [
 'Quantum mechanics describes the behavior of very small particles.',
 'Photosynthesis is the process by which green plants make food using sunlight.',
 'Artificial Intelligence aims to create machines that can think and learn.',
 'The pyramids of Egypt are historical monuments that have stood for thousands of years.',
 'New Delhi is the capital of India and the seat of all three branches of the Government of India.',
 'Biology is the study of living organisms and their interactions with the environment.',
 'Music therapy can aid in the mental well-being of individuals.',
 'Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.',
 'The Milky Way is just one of billions of galaxies in the universe.',
 'Economic theories help understand the distribution of resources in society.',
 'Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.',
 'Yoga is an ancient practice that involves physical postures and meditation.'
]

In [23]:
from langchain_chroma import Chroma

# create vector DB of docs and embeddings - takes 1 min on Colab
chroma_db = Chroma.from_texts(texts=docs, collection_name='db_docs',
                              # need to set the distance function to cosine else it uses euclidean by default
                              # check https://docs.trychroma.com/guides#changing-the-distance-function
                              collection_metadata={"hnsw:space": "cosine"},
                              embedding=openai_embed_model)

In [24]:
similarity_threshold_retriever = chroma_db.as_retriever(search_type="similarity_score_threshold",
                                                        search_kwargs={"k": 3,
                                                                       "score_threshold": 0.3})

In [25]:
query = "what is the capital of India?"
top3_docs = similarity_threshold_retriever.invoke(query)
top3_docs

[Document(id='ca506f7e-af35-47e3-94c5-4ffdd5c1bf8a', metadata={}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.'),
 Document(id='df9d7443-b616-4d3c-ac59-a240c3be551c', metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.'),
 Document(id='d098069b-c6a3-401d-a03d-59aafa0046b2', metadata={}, page_content='Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.')]

In [26]:
from langchain_core.prompts import ChatPromptTemplate

prompt = """You are an assistant for question-answering tasks.
            Use the following pieces of retrieved context to answer the question.
            If no context is present or if you don't know the answer, just say that you don't know.
            Do not make up the answer unless it is there in the provided context.
            Keep the answer concise and to the point with regard to the question.

            Question:
            {question}

            Context:
            {context}

            Answer:
         """

prompt_template = ChatPromptTemplate.from_template(prompt)

In [27]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [28]:


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_rag_chain = (
    {
        "context": (similarity_threshold_retriever | RunnableLambda(format_docs)), # first retriever will retrive the node then it goes to format_docs function that 
                                                            # will get the page content from each retrived document(page1  content /n/n page2 content)
        "question": RunnablePassthrough()
    }
      |
    prompt_template
      |
    chatgpt
)

In [29]:
query = "What is the capital of India?"
result = qa_rag_chain.invoke(query)
print(result.content)

New Delhi is the capital of India.
